In [1]:
import os
import zipfile
import urllib.request
import arcpy

In [2]:
# Radni folder
workspace = r"C:\fakultet\osm"
zip_path = os.path.join(workspace, "serbia-latest-free.shp.zip")
extracted_folder = os.path.join(workspace, "serbia_shp")

arcpy.env.workspace = extracted_folder
arcpy.env.overwriteOutput = True

In [4]:
# URL za preuzimanje OSM podataka
url = "https://download.geofabrik.de/europe/serbia-latest-free.shp.zip"


In [5]:
# Preuzimanje fajla
if not os.path.exists(zip_path):
    print("Preuzimam OSM podatke za Srbiju...")
    urllib.request.urlretrieve(url, zip_path)
    print("Preuzimanje završeno.")
else:
    print("Zip fajl već postoji, preskačem preuzimanje.")


Zip fajl već postoji, preskačem preuzimanje.


In [6]:
# Raspakivanje
if not os.path.exists(extracted_folder):
    os.makedirs(extracted_folder)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extracted_folder)

print("Podaci raspakovani.")

Podaci raspakovani.


In [28]:
# uzme prvu mapu u projektu
aprx = arcpy.mp.ArcGISProject("CURRENT")
mapa = aprx.listMaps()[0]   

In [7]:
# Putanja do layer-a opština
granice_opstina = os.path.join(extracted_folder, "gis_osm_places_a_free_1.shp")

# Opština koju želimo analizirati
search_text = "Врачар"   
temp_layer = "opstina_layer"

# Kreiranje privremenog feature layer-a za Clip
arcpy.management.MakeFeatureLayer(
    granice_opstina, temp_layer, f"NAME LIKE '%{search_text}%'"
)

# Provera da li postoji poligon opštine
count = int(arcpy.management.GetCount(temp_layer)[0])
if count == 0:
    raise ValueError(f"Nije pronađen poligon koji sadrži '{search_text}' u {granice_opstina}.")
else:
    print(f"Pronađen poligon koji sadrži '{search_text}'.")

Pronađen poligon koji sadrži 'Врачар'.


In [8]:
# Lista OSM layer-a koje želimo da isečemo
feature_classes = [
    "gis_osm_roads_free_1.shp",
    "gis_osm_buildings_a_free_1.shp",
    "gis_osm_landuse_a_free_1.shp",
    "gis_osm_water_a_free_1.shp",
    "gis_osm_natural_a_free_1.shp"
]

# Promena layer-a u srpske nazive
srpski_nazivi = {
    "gis_osm_roads_free_1.shp": "putevi.shp",
    "gis_osm_buildings_a_free_1.shp": "zgrade.shp",
    "gis_osm_landuse_a_free_1.shp": "koriscenje_zemljista.shp",
    "gis_osm_water_a_free_1.shp": "vode.shp",
    "gis_osm_natural_a_free_1.shp": "prirodni_elementi.shp"
}

# Folder za isečene layer-e
output_folder = os.path.join(workspace, "vracar_clip")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Sečenje i čuvanje
for fc in feature_classes:
    in_feature = os.path.join(extracted_folder, fc)
    if not arcpy.Exists(in_feature):
        print(f"Fajl {fc} ne postoji, preskačem.")
        continue

    out_feature = os.path.join(output_folder, srpski_nazivi[fc])
    print(f"Sečem {fc} na teritoriju Vračara...")
    arcpy.analysis.Clip(in_feature, temp_layer, out_feature)
    print(f"Sačuvano: {out_feature}")

print("Svi slojevi su isečeni i preimenovani.")

Sečem gis_osm_roads_free_1.shp na teritoriju Vračara...
Sačuvano: C:\fakultet\osm\vracar_clip\putevi.shp
Sečem gis_osm_buildings_a_free_1.shp na teritoriju Vračara...
Sačuvano: C:\fakultet\osm\vracar_clip\zgrade.shp
Sečem gis_osm_landuse_a_free_1.shp na teritoriju Vračara...
Sačuvano: C:\fakultet\osm\vracar_clip\koriscenje_zemljista.shp
Sečem gis_osm_water_a_free_1.shp na teritoriju Vračara...
Sačuvano: C:\fakultet\osm\vracar_clip\vode.shp
Sečem gis_osm_natural_a_free_1.shp na teritoriju Vračara...
Sačuvano: C:\fakultet\osm\vracar_clip\prirodni_elementi.shp
Svi slojevi su isečeni i preimenovani.


In [9]:
# Putanja do isečenih puteva
putevi = os.path.join(output_folder, "putevi.shp")

In [10]:
# Dodavanje kolone sa dužinom puta
if "DUZINA_m" not in [f.name for f in arcpy.ListFields(putevi)]:
    arcpy.management.AddField(putevi, "DUZINA_m", "DOUBLE")

# Izračunavanje geometrije
arcpy.management.CalculateGeometryAttributes(
    putevi, [["DUZINA_m", "LENGTH_GEODESIC"]], length_unit="METERS"
)
print("Kolona 'DUZINA_m' dodata i izračunata.")

Kolona 'DUZINA_m' dodata i izračunata.


In [11]:
# Dodavanje kolone sa tipom puta
if "tip_puta" not in [f.name for f in arcpy.ListFields(putevi)]:
    arcpy.management.AddField(putevi, "tip_puta", "TEXT", field_length=50)

# Promena layer-a u srpske nazive
prevod = {
    "motorway": "Autoput",
    "trunk": "Brzi put",
    "primary": "Magistralni put",
    "primary_link": "Veza na magistralni put",
    "secondary": "Regionalni put",
    "secondary_link": "Veza na regionalni put",
    "tertiary": "Lokalni put",
    "residential": "Ulica u naselju",
    "living_street": "Zona stanovanja",
    "unclassified": "Nerazvrstani put",
    "service": "Servisni put",
    "track": "Poljski/šumski put",
    "track_grade5": "Poljski/šumski put (lošeg kvaliteta)",
    "footway": "Pešačka staza",
    "pedestrian": "Pešačka ulica",
    "path": "Staza",
    "steps": "Stepenice",
    "cycleway": "Biciklistička staza"
}

with arcpy.da.UpdateCursor(putevi, ["fclass", "tip_puta"]) as cursor:
    for row in cursor:
        row[1] = prevod.get(row[0], "Nepoznato")
        cursor.updateRow(row)

print("Kolona 'tip_puta' popunjena prema fclass.")


Kolona 'tip_puta' popunjena prema fclass.


In [12]:
# Dissolve analiza – dužine po tipu puta
roads_dissolve = os.path.join(output_folder, "saobracajnice_vracar_dissolve.shp")

arcpy.management.Dissolve(
    putevi,
    roads_dissolve,
    ["tip_puta"],
    [["DUZINA_m", "SUM"]]
)
print("Dissolve je izračunat.")


Dissolve je izračunat.


In [13]:
# Dodavanje procenta
if "PROCENAT" not in [f.name for f in arcpy.ListFields(roads_dissolve)]:
    arcpy.management.AddField(roads_dissolve, "PROCENAT", "DOUBLE")

total_length = 0
with arcpy.da.SearchCursor(roads_dissolve, ["SUM_DUZINA"]) as cursor:
    for row in cursor:
        total_length += row[0]

with arcpy.da.UpdateCursor(roads_dissolve, ["SUM_DUZINA", "PROCENAT"]) as cursor:
    for row in cursor:
        percent_value = (row[0] / total_length) * 100 if total_length > 0 else 0
        row[1] = percent_value
        cursor.updateRow(row)

print("Kolona 'PROCENAT' je dodata.")

Kolona 'PROCENAT' je dodata.


In [14]:
# POI analiza
pois = os.path.join(extracted_folder, "gis_osm_pois_a_free_1.shp")
pois_points = os.path.join(output_folder, "pois_points.shp")
pois_clip = os.path.join(output_folder, "pois_vracar.shp")


In [15]:
# Pretvaranje POI poligona u tačke
arcpy.management.FeatureToPoint(pois, pois_points, "INSIDE")
print("POI pretvoren u tačke: pois_points.shp")

POI pretvoren u tačke: pois_points.shp


In [16]:
# Isecanje tačaka po granici opštine Vračar
arcpy.analysis.Clip(pois_points, temp_layer, pois_clip)
print("POI isečen po granici Vračara: pois_vracar.shp")


POI isečen po granici Vračara: pois_vracar.shp


In [17]:
# Prevod POI fclass na srpski
poi_prevod = {
    "park": "Park",
    "dog_park": "Park za pse",
    "pitch": "Sportski teren",
    "stadium": "Stadion",
    "sports_centre": "Sportski centar",
    "playground": "Igralište",
    "community_centre": "Mesna zajednica",
    "library": "Biblioteka",
    "theatre": "Pozorište",
    "cinema": "Bioskop",
    "restaurant": "Restoran",
    "cafe": "Kafić",
    "fast_food": "Brza hrana",
    "bar": "Bar",
    "pub": "Pab",
    "hotel": "Hotel",
    "motel": "Motel",
    "guest_house": "Pansion",
    "hotel school": "Ugostiteljska škola",
    "supermarket": "Supermarket",
    "convenience": "Prodavnica",
    "mall": "Tržni centar",
    "market_place": "Pijaca",
    "florist": "Cvećara",
    "kiosk": "Kiosk",
    "toilet": "Javni WC",
    "fountain": "Fontana",
    "shelter": "Sklonište",
    "tower": "Toranj",
    "tourist_info": "Turistički info",
    "embassy": "Ambasada",
    "courthouse": "Sud",
    "police": "Policija",
    "clinic": "Klinika",
    "hospital": "Bolnica",
    "pharmacy": "Apoteka",
    "water_works": "Vodovod",
    "school": "Škola",
    "kindergarten": "Vrtić",
    "university": "Univerzitet"
}


In [18]:
# Dodavanje kolone za prevod
if "tip_poi" not in [f.name for f in arcpy.ListFields(pois_clip)]:
    arcpy.management.AddField(pois_clip, "tip_poi", "TEXT", field_length=100)


In [19]:
# Popunjavanje kolone za prevod
with arcpy.da.UpdateCursor(pois_clip, ["fclass", "tip_poi"]) as cursor:
    for row in cursor:
        row[1] = poi_prevod.get(row[0], "Nepoznato")
        cursor.updateRow(row)

print("fclass preveden na srpski u koloni 'tip_poi'.")

fclass preveden na srpski u koloni 'tip_poi'.


In [23]:
# Ažuriranje sloja u mapi na srpski
layer_pois = mapa.addDataFromPath(pois_clip)
sym_pois = layer_pois.symbology
sym_pois.updateRenderer('UniqueValueRenderer')
sym_pois.renderer.fields = ["tip_poi"]
layer_pois.symbology = sym_pois
layer_pois.visible = True

In [24]:
# Dodavanje dissolve puteva
layer_roads = mapa.addDataFromPath(roads_dissolve)


In [25]:
# Dodavanje poligona opštine
layer_opstina = mapa.addDataFromPath(granice_opstina)


In [26]:
# Putevi – simbologija po procentu
sym_roads = layer_roads.symbology
sym_roads.updateRenderer('GraduatedColorsRenderer')
sym_roads.renderer.classificationField = "PROCENAT"
sym_roads.renderer.breakCount = 5
sym_roads.renderer.colorRamp = aprx.listColorRamps("Blue-Green (Continuous)")[0]
layer_roads.symbology = sym_roads

In [27]:
# Opština – simbologija
sym_poly = layer_opstina.symbology
sym_poly.updateRenderer('SimpleRenderer')
sym_poly.renderer.symbol.color = {'RGB': [173, 216, 230, 100]}
layer_opstina.symbology = sym_poly

print("Svi slojevi su dodati.")

Svi slojevi su dodati.
